In [3]:
import os
import glob
import json
import re
import numpy as np
import sys
import datetime

%run unify_data_format

Found 19282 meta data files in ['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202405_202406_GXe_threshold/', '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202405_202409_GXe', '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202409_tests', '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe', '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe_new']


In [2]:
# DATA_FOLDERS = [
#     "/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/"
# ]


# for data_folder in DATA_FOLDERS:
#     if not os.path.isabs(data_folder) and not os.path.isdir(data_folder):
#         raise ValueError(f"{data_folder} is not an absolute path for a directory")

# # Get the list of data files from DATA_FOLDERS
# data_files_name = get_data_files(DATA_FOLDERS, pattern="meta_config*.json")
# print(f"Found {len(data_files_name)} meta data files in {DATA_FOLDERS}")

# file_name = v_basename(data_files_name)
# dir_name = v_dirname(data_files_name)
# test_name = v_get_test_name(data_files_name)

# #turn all list into np.array
# data_files_name = np.array(data_files_name) # path + file name
# file_name = np.array(file_name)
# dir_name = np.array(dir_name)

In [4]:
test_set_list = step0_find_problems()

Found 19282 meta data files in ['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202405_202406_GXe_threshold/', '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202405_202409_GXe', '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202409_tests', '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe', '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe_new']


In [6]:
unique, unique_counts = np.unique(test_set_list, return_counts=1)
# unique[unique_counts!=24], unique_counts[unique_counts!=24]
unique, unique_counts

(array(['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241009_T98_48V_4.0sig/threshold_calibration|20241010_095823.json',
        '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241009_T98_48V_4.0sig|20241010_095831.json',
        '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241009_T98_48V_4.5sig|20241010_100344.json',
        '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241009_T98_48V_5.0sig|20241010_101234.json',
        '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241009_T98_48V_5.5sig|20241010_102614.json',
        '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241010_2_T98_47V_2.5sig/threshold_calibration|20241010_210239.json',
        '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241010_2_T98_47V_2.5sig/threshold_calibration|20241010_211026.json',
      

In [7]:
# ### case by case

setname_list = np.char.split(unique,sep="|")
setname_list

array([list(['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241009_T98_48V_4.0sig/threshold_calibration', '20241010_095823.json']),
       list(['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241009_T98_48V_4.0sig', '20241010_095831.json']),
       list(['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241009_T98_48V_4.5sig', '20241010_100344.json']),
       list(['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241009_T98_48V_5.0sig', '20241010_101234.json']),
       list(['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241009_T98_48V_5.5sig', '20241010_102614.json']),
       list(['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241010_2_T98_47V_2.5sig/threshold_calibration', '20241010_210239.json']),
       list(['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data/202410_LXe/20241010_2_T

In [9]:
len(setname_list)

115

In [10]:
file_in_set_arr = np.array([])

for setname in setname_list[:]:
    set_files_full_path = get_data_files(DATA_FOLDERS, pattern=f"meta_config*{setname[1]}")
    
    board_0_list = np.array([])
    board_1_list = np.array([])
    board_id = None
    
    breakpoint = 0 
    
    # loop over the 24 files
    for i in range(len(set_files_full_path)):
        
        if board_id != 0:
            break
        
        full_path = set_files_full_path[i]
        # print(f"Processing {full_path}")

        with open(full_path, "r") as file:
            meta_data = json.load(file)
            
        if meta_data["channel"]<16:
            continue
        else:
            if ("board_0_channels" in meta_data):
                board_id = meta_data["board_0_channels"][0]
                # print(meta_data["board_0_channels"])
                # board_0_list = np.append(board_0_list, meta_data["board_0_channels"][0])
            elif ("board_1_channels" in meta_data):
                # board_1_list = np.append(board_1_list, meta_data["board_1_channels"][0])
                board_id = meta_data["board_1_channels"][0]
                
            if (meta_data["channel"]==16) and (board_id==0):
                breakpoint = 16
            elif (meta_data["channel"]==16) and (board_id==4):
                breakpoint = 12
                
            elif (meta_data["channel"]==17) and (board_id==1):
                breakpoint = 16
            elif (meta_data["channel"]==17) and (board_id==5):
                breakpoint = 12
                
            elif (meta_data["channel"]==18) and (board_id==2):
                breakpoint = 16
            elif (meta_data["channel"]==18) and (board_id==6):
                breakpoint = 12
                
            elif (meta_data["channel"]==19) and (board_id==3):
                breakpoint = 16
            elif (meta_data["channel"]==19) and (board_id==7):
                breakpoint = 12
                
            elif (meta_data["channel"]==20) and (board_id==4):
                breakpoint = 16
            elif (meta_data["channel"]==20) and (board_id==8):
                breakpoint = 12
                
            elif (meta_data["channel"]==21) and (board_id==5):
                breakpoint = 16
            elif (meta_data["channel"]==21) and (board_id==9):
                breakpoint = 12
                
            elif (meta_data["channel"]==22) and (board_id==6):
                breakpoint = 16
            elif (meta_data["channel"]==22) and (board_id==10):
                breakpoint = 12
                
            elif (meta_data["channel"]==23) and (board_id==7):
                breakpoint = 16
            elif (meta_data["channel"]==23) and (board_id==11):
                breakpoint = 12
            
            elif (meta_data["channel"]==12) and (board_id==0):
                breakpoint = 12
            elif (meta_data["channel"]==12) and (board_id==12):
                breakpoint = 16
                
            elif (meta_data["channel"]==13) and (board_id==1):
                breakpoint = 12
            elif (meta_data["channel"]==13) and (board_id==13):
                breakpoint = 16
            
            elif (meta_data["channel"]==14) and (board_id==2):
                breakpoint = 12
            elif (meta_data["channel"]==14) and (board_id==14):
                breakpoint = 16
            
            elif (meta_data["channel"]==15) and (board_id==3):
                breakpoint = 12
            elif (meta_data["channel"]==15) and (board_id==15):
                breakpoint = 16
                
    if breakpoint == 0: # meaning that no files above 12 to determine the breakpoint -> not so important -> default case: 16
        breakpoint = 16
        
    if breakpoint == 12:
        board_0_list = np.arange(0,12)
        board_1_list = np.arange(12,24)
    elif breakpoint == 16:
        board_0_list = np.arange(0,16)
        board_1_list = np.arange(16,24)
    else: 
        raise ValueError
    
    print(board_0_list, board_1_list)

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] [16 17 18 19 20 21 22 23]
[ 0  1  2  3

In [6]:
date = "20241101"
time = "112644"
# "start_timestamp": "2024-11-01 11:26:44.985009",


In [ ]:
year = date[:4]
month = date[4:6]
day = date[6:]

hour = time[:2]
minute = time[2:4]
second = time[4:]

time_str = f'{year}-{month}-{day} {hour}:{minute}:{float(second):.6f}'
time_str

'2024-11-01 11:26:44.000000'

In [ ]:
pd.to_datetime(_tmp, format="%Y-%m-%d %H:%M:%S.%f")

In [31]:
def gen_new_name(path):
    path = path.replace("tmp", "DAQ_config")
    return path

# vectorize the function
v_gen_new_name = np.vectorize(gen_new_name, otypes=[np.ndarray])

In [72]:
def rename_dir(current_dir, new_dir):
    os.rename(current_dir, new_dir)
    return


In [38]:
root_path = "/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new"


In [74]:
config_paths = glob.glob(os.path.join(root_path, "**","tmp/"), recursive=True)

config_paths = np.array(config_paths)
config_paths


array(['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240924_T102_49V_5.0sig/tmp/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_2_T102_49V_6.0sig/tmp/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_2_T102_49V_10.0sig/tmp/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_3_T102_49V_7.0sig/tmp/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_4_T102_49V_5.0sig/tmp/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_5_T102_49V_10.0sig/tmp/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_6_T102_49V_7.0sig/tmp/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20

In [75]:
len(config_paths)

480

In [76]:
new_config_paths = v_gen_new_name(config_paths)
new_config_paths

array(['/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240924_T102_49V_5.0sig/DAQ_config/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_2_T102_49V_6.0sig/DAQ_config/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_2_T102_49V_10.0sig/DAQ_config/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_3_T102_49V_7.0sig/DAQ_config/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_4_T102_49V_5.0sig/DAQ_config/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_5_T102_49V_10.0sig/DAQ_config/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/softlink_to_data/all_data_new/202409_tests/test_20240926_6_T102_49V_7.0sig/DAQ_config/',
       '/home/daqtest/DAQ/SandyAQ_vera/SandyAQ/s

In [77]:
len(new_config_paths)

480

In [79]:
assert len(config_paths) == len(new_config_paths)

for i in range(len(config_paths)):
    rename_dir(config_paths[i], new_config_paths[i])